## Extração Features

In [ ]:


# Caminho para o arquivo de features
# O notebook está em 'Notebooks/', e o arquivo em 'Notebooks/extracted_features/'
file_path = 'extracted_features/rte_features.pt'

# Carregar as informações salvas
if os.path.exists(file_path):
    saved_data = torch.load(file_path)

    # Exibir as chaves do dicionário para verificar
    print("Informações carregadas do arquivo:")
    print(saved_data.keys())

    # Acessar as informações
    bert_features = saved_data['bert_features']
    bert_labels = saved_data['bert_labels']
    roberta_features = saved_data['roberta_features']
    roberta_labels = saved_data['roberta_labels']
    dataset_info = saved_data['dataset_info']

dataset_info

Informações carregadas do arquivo:
dict_keys(['bert_features', 'bert_labels', 'roberta_features', 'roberta_labels', 'dataset_info'])


{'name': 'RTE',
 'size': 2490,
 'bert_layer': 'bert.pooler',
 'roberta_layer': 'classifier.dense'}

torch.Size([2490, 768])

In [3]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from pymfe.mfe import MFE
import torch

def to_numpy(x):
    if isinstance(x, np.ndarray):
        return x
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def ensure_dir(path: str | Path) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)
    
pd.set_option('display.max_rows', None)

In [4]:
print("Groups:", MFE.valid_groups())
print("Some features:", sorted(list(MFE.valid_metafeatures()))[:40])
print("Summaries:", MFE.valid_summary())

Groups: ('landmarking', 'general', 'statistical', 'model-based', 'info-theory', 'relative', 'clustering', 'complexity', 'itemset', 'concept')
Some features: ['attr_conc', 'attr_ent', 'attr_to_inst', 'best_node', 'c1', 'c2', 'can_cor', 'cat_to_num', 'ch', 'class_conc', 'class_ent', 'cls_coef', 'cohesiveness', 'conceptvar', 'cor', 'cov', 'density', 'eigenvalues', 'elite_nn', 'eq_num_attr', 'f1', 'f1v', 'f2', 'f3', 'f4', 'freq_class', 'g_mean', 'gravity', 'h_mean', 'hubs', 'impconceptvar', 'inst_to_attr', 'int', 'iq_range', 'joint_ent', 'kurtosis', 'l1', 'l2', 'l3', 'leaves']
Summaries: ('mean', 'nanmean', 'sd', 'nansd', 'var', 'nanvar', 'count', 'nancount', 'histogram', 'nanhistogram', 'iq_range', 'naniq_range', 'kurtosis', 'nankurtosis', 'max', 'nanmax', 'median', 'nanmedian', 'min', 'nanmin', 'quantiles', 'nanquantiles', 'range', 'nanrange', 'skewness', 'nanskewness', 'sum', 'nansum', 'powersum', 'pnorm', 'nanpowersum', 'nanpnorm')


### Statistical Features

In [25]:
X = bert_features
y = np.asarray(bert_labels).astype(int).ravel()

try:
    import torch
    X = X.detach().cpu().numpy() if isinstance(X, torch.Tensor) else np.asarray(X)
except Exception:
    X = np.asarray(X)
n, d = X.shape
print(f"bert_features shape = {X.shape}")

# Padronização
X_std = StandardScaler().fit_transform(X)

try:
    stat_feats = set(MFE.valid_metafeatures(groups=["statistical"]))
    if not stat_feats:
        stat_feats = set(MFE.valid_metafeatures())
except TypeError:
    stat_feats = set(MFE.valid_metafeatures())
feats = sorted(list(stat_feats))
if len(feats) == 0:
    raise RuntimeError("no features available!")

try:
    valid_sum = set(MFE.valid_summary())
except Exception:
    valid_sum = {"mean","sd","median","min","max"}
wanted_sum = ["mean","sd","median","min","max","quantiles"]
summaries = [s for s in wanted_sum if s in valid_sum]
if not summaries:
    summaries = ["mean","sd"]

print(f"[PyMFE] {len(feats)} features estatísticas | summaries={summaries}")


bert_features shape = (2490, 768)
[PyMFE] 29 features estatísticas | summaries=['mean', 'sd', 'median', 'min', 'max', 'quantiles']


In [7]:
mfe = MFE(groups=["statistical"], features=feats, summary=summaries, random_state=42)
mfe.fit(X_std)                
names, values = mfe.extract()

df_stat_bert = pd.Series(dict(zip(names, values)), dtype="float64") \
                  .rename_axis("feature").to_frame("value").reset_index()
df_stat_bert["dataset"], df_stat_bert["split"], df_stat_bert["repr"] = "RTE", "train", "bert_features"
df_stat_bert["n"], df_stat_bert["d"] = int(n), int(d)


out_dir = Path("meta"); out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "RTE_train_bert_features_stats.parquet"
df_stat_bert.to_parquet(out_path, index=False)
print(f"✔ BERT stats → {out_path.resolve()}  ({len(df_stat_bert)} linhas)")

df_stat_bert[df_stat_bert["value"].notna()]

/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:1568: UserWarning: It is not possible make equal discretization
  warnings.warn("It is not possible make equal discretization")
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'g_mean' with summary 'min'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'g_mean' with summary 'sd'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'g_mean' with summary 'median'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'g_mean' with summary 'max'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe

✔ BERT stats → /home/dods/matrix/IC/Notebooks/meta/RTE_train_bert_features_stats.parquet  (183 linhas)


/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:606: RuntimeWarning: Can't extract feature 'var'.
 Exception message: AttributeError('`np.asfarray` was removed in the NumPy 2.0 release. Use `np.asarray` with a proper dtype instead.').
 Will set it as 'np.nan' for all summary functions.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'var' with summary 'min'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'var' with summary 'sd'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'var' with summary 'median'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'var' wi

,feature,value,dataset,split,repr,n,d
0,cor.max,9.998142e-01,RTE,train,bert_features,2490,768
1,cor.mean,5.008932e-01,RTE,train,bert_features,2490,768
2,cor.median,5.177412e-01,RTE,train,bert_features,2490,768
3,cor.min,3.067453e-06,RTE,train,bert_features,2490,768
4,cor.quantiles.0,3.067453e-06,RTE,train,bert_features,2490,768
5,cor.quantiles.1,3.619151e-01,RTE,train,bert_features,2490,768
6,cor.quantiles.2,5.177412e-01,RTE,train,bert_features,2490,768
7,cor.quantiles.3,6.579655e-01,RTE,train,bert_features,2490,768
8,cor.quantiles.4,9.998142e-01,RTE,train,bert_features,2490,768
9,cor.sd,2.067046e-01,RTE,train,bert_features,2490,768


### Complexity Meta-features

In [8]:
try:
    comp_valid = set(MFE.valid_metafeatures(groups=["complexity"]))
    if not comp_valid:
        comp_valid = set(MFE.valid_metafeatures())
except TypeError:
    comp_valid = set(MFE.valid_metafeatures())

comp_valid

{'c1',
 'c2',
 'cls_coef',
 'density',
 'f1',
 'f1v',
 'f2',
 'f3',
 'f4',
 'hubs',
 'l1',
 'l2',
 'l3',
 'lsc',
 'n1',
 'n2',
 'n3',
 'n4',
 't1',
 't2',
 't3',
 't4'}

In [26]:
priority_complex = {"f1","f2","f3","f4","n1","n2","n3","n4","l1","l2", "l3","t1","t2","t3","t4"}
feats_comp = sorted(list(priority_complex & comp_valid))
if not feats_comp:
    feats_comp = sorted(list(comp_valid))

summaries_comp = ["mean"]

mfe_c = MFE(groups=["complexity"], features=feats_comp, summary=summaries_comp, random_state=42)
mfe_c.fit(X_std, y)
names_c, values_c = mfe_c.extract()

df_complex_bert = pd.Series(dict(zip(names_c, values_c)), dtype="float64").rename_axis("feature").to_frame("value").reset_index()

df_complex_bert

/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:1568: UserWarning: It is not possible make equal discretization
  warnings.warn("It is not possible make equal discretization")
/home/dods/miniconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:606: RuntimeWarning: Can't extract feature 'n1'.
 Exception message: AttributeError('`np.asfarray` was removed in the NumPy 2.0 release. Use `np.asarray` with a proper dtype instead.').
  warn

,feature,value
0,f1.mean,9.993685e-01
1,f2.mean,1.072554e-60
2,f3.mean,9.943775e-01
3,f4.mean,0.000000e+00
4,l1.mean,1.553515e-01
5,l2.mean,2.036145e-01
6,l3.mean,4.224900e-01
7,n1,NaN
8,n2.mean,4.980219e-01
9,n3.mean,4.706827e-01


### General Meta-Features

In [10]:
try:
    gen_valid = set(MFE.valid_metafeatures(groups=["general"]))
    if not gen_valid:
        gen_valid = set(MFE.valid_metafeatures())
except TypeError:
    gen_valid = set(MFE.valid_metafeatures())

gen_valid

{'attr_to_inst',
 'cat_to_num',
 'freq_class',
 'inst_to_attr',
 'nr_attr',
 'nr_bin',
 'nr_cat',
 'nr_class',
 'nr_inst',
 'nr_num',
 'num_to_cat'}

In [11]:
priority_general = {"nr_inst","nr_attr","attr_to_inst","nr_class","class_ent"}
feats_gen = sorted(list(priority_general & gen_valid))
if not feats_gen:
    feats_gen = sorted(list(gen_valid))

summaries_gen = ["mean"]

mfe_g = MFE(groups=["general"], features=feats_gen, summary=summaries_gen, random_state=42)
mfe_g.fit(X_std, y)
names_g, values_g = mfe_g.extract()

df_general_bert = pd.Series(dict(zip(names_g, values_g)), dtype="float64") \
    .rename_axis("feature").to_frame("value").reset_index()

display(df_general_bert.head(10))

/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:1568: UserWarning: It is not possible make equal discretization
  warnings.warn("It is not possible make equal discretization")


,feature,value
0,attr_to_inst,0.308434
1,nr_attr,768.000000
2,nr_class,2.000000
3,nr_inst,2490.000000


In [12]:
p_complex = Path("meta") / "RTE_train_bert_features_complex.parquet"
p_complex.parent.mkdir(parents=True, exist_ok=True)
df_complex_bert.to_parquet(p_complex, index=False)
print(f"✔ BERT complexidade → {p_complex.resolve()}  ({len(df_complex_bert)} linhas)")

p_general = Path("meta") / "RTE_train_bert_features_general.parquet"
p_general.parent.mkdir(parents=True, exist_ok=True)
df_general_bert.to_parquet(p_general, index=False)
print(f"✔ BERT gerais → {p_general.resolve()}  ({len(df_general_bert)} linhas)")


✔ BERT complexidade → /home/dods/matrix/IC/Notebooks/meta/RTE_train_bert_features_complex.parquet  (12 linhas)
✔ BERT gerais → /home/dods/matrix/IC/Notebooks/meta/RTE_train_bert_features_general.parquet  (4 linhas)


## Roberta Features

In [27]:
X = roberta_features
y = np.asarray(roberta_labels).astype(int).ravel()

try:
    import torch
    X = X.detach().cpu().numpy() if isinstance(X, torch.Tensor) else np.asarray(X)
except Exception:
    X = np.asarray(X)
n, d = X.shape
print(f"bert_features shape = {X.shape}")

# Padronização
X_std = StandardScaler().fit_transform(X)

try:
    stat_feats = set(MFE.valid_metafeatures(groups=["statistical"]))
    if not stat_feats:
        stat_feats = set(MFE.valid_metafeatures())
except TypeError:
    stat_feats = set(MFE.valid_metafeatures())
feats = sorted(list(stat_feats))
if len(feats) == 0:
    raise RuntimeError("no features available!")

try:
    valid_sum = set(MFE.valid_summary())
except Exception:
    valid_sum = {"mean","sd","median","min","max"}
wanted_sum = ["mean","sd","median","min","max","quantiles"]
summaries = [s for s in wanted_sum if s in valid_sum]
if not summaries:
    summaries = ["mean","sd"]

print(f"[PyMFE] {len(feats)} features estatísticas | summaries={summaries}")


bert_features shape = (2490, 768)
[PyMFE] 29 features estatísticas | summaries=['mean', 'sd', 'median', 'min', 'max', 'quantiles']


In [16]:
mfe = MFE(groups=["statistical"], features=feats, summary=summaries, random_state=42)
mfe.fit(X_std)                
names, values = mfe.extract()

df_stat_bert = pd.Series(dict(zip(names, values)), dtype="float64") \
                  .rename_axis("feature").to_frame("value").reset_index()
df_stat_bert["dataset"], df_stat_bert["split"], df_stat_bert["repr"] = "RTE", "train", "bert_features"
df_stat_bert["n"], df_stat_bert["d"] = int(n), int(d)


out_dir = Path("meta"); out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "RTE_train_bert_features_stats.parquet"
df_stat_bert.to_parquet(out_path, index=False)
print(f"✔ BERT stats → {out_path.resolve()}  ({len(df_stat_bert)} linhas)")

print(df_stat_bert[df_stat_bert["value"].notna()])

/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'g_mean' with summary 'min'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'g_mean' with summary 'sd'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'g_mean' with summary 'median'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'g_mean' with summary 'max'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'g_mean' with summary 'quantiles'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_inte

✔ BERT stats → /home/dods/matrix/IC/Notebooks/meta/RTE_train_bert_features_stats.parquet  (183 linhas)
                     feature         value dataset  split           repr  \
0                    cor.max  6.230767e-01     RTE  train  bert_features   
1                   cor.mean  1.074198e-01     RTE  train  bert_features   
2                 cor.median  9.091321e-02     RTE  train  bert_features   
3                    cor.min  7.167637e-07     RTE  train  bert_features   
4            cor.quantiles.0  7.167637e-07     RTE  train  bert_features   
5            cor.quantiles.1  4.312116e-02     RTE  train  bert_features   
6            cor.quantiles.2  9.091321e-02     RTE  train  bert_features   
7            cor.quantiles.3  1.550964e-01     RTE  train  bert_features   
8            cor.quantiles.4  6.230767e-01     RTE  train  bert_features   
9                     cor.sd  8.083074e-02     RTE  train  bert_features   
10                   cov.max  6.233270e-01     RTE  train  be

/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:606: RuntimeWarning: Can't extract feature 'var'.
 Exception message: AttributeError('`np.asfarray` was removed in the NumPy 2.0 release. Use `np.asarray` with a proper dtype instead.').
 Will set it as 'np.nan' for all summary functions.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'var' with summary 'min'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'var' with summary 'sd'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'var' with summary 'median'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'var' wi

In [28]:
priority_complex = {"f1","f2","f3","f4","n1","n2","n3","n4","l1","l2", "l3","t1","t2","t3","t4"}
feats_comp = sorted(list(priority_complex & comp_valid))
if not feats_comp:
    feats_comp = sorted(list(comp_valid))

summaries_comp = ["mean"]

mfe_c = MFE(groups=["complexity"], features=feats_comp, summary=summaries_comp, random_state=42)
mfe_c.fit(X_std, y)
names_c, values_c = mfe_c.extract()

df_complex_bert = pd.Series(dict(zip(names_c, values_c)), dtype="float64").rename_axis("feature").to_frame("value").reset_index()

df_complex_bert

/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:606: RuntimeWarning: Can't extract feature 'n1'.
 Exception message: AttributeError('`np.asfarray` was removed in the NumPy 2.0 release. Use `np.asarray` with a proper dtype instead.').
  warnings.warn(


,feature,value
0,f1.mean,9.990899e-01
1,f2.mean,3.789245e-42
2,f3.mean,9.939759e-01
3,f4.mean,0.000000e+00
4,l1.mean,1.421526e-01
5,l2.mean,1.807229e-01
6,l3.mean,4.377510e-01
7,n1,NaN
8,n2.mean,4.958394e-01
9,n3.mean,4.578313e-01


In [18]:
priority_general = {"nr_inst","nr_attr","attr_to_inst","nr_class","class_ent"}
feats_gen = sorted(list(priority_general & gen_valid))
if not feats_gen:
    feats_gen = sorted(list(gen_valid))

summaries_gen = ["mean"]

mfe_g = MFE(groups=["general"], features=feats_gen, summary=summaries_gen, random_state=42)
mfe_g.fit(X_std, y)
names_g, values_g = mfe_g.extract()

df_general_bert = pd.Series(dict(zip(names_g, values_g)), dtype="float64") \
    .rename_axis("feature").to_frame("value").reset_index()

display(df_general_bert.head(10))

,feature,value
0,attr_to_inst,0.308434
1,nr_attr,768.000000
2,nr_class,2.000000
3,nr_inst,2490.000000


In [19]:
p_complex = Path("meta") / "RTE_train_bert_features_complex.parquet"
p_complex.parent.mkdir(parents=True, exist_ok=True)
df_complex_bert.to_parquet(p_complex, index=False)
print(f"✔ BERT complexidade → {p_complex.resolve()}  ({len(df_complex_bert)} linhas)")

p_general = Path("meta") / "RTE_train_bert_features_general.parquet"
p_general.parent.mkdir(parents=True, exist_ok=True)
df_general_bert.to_parquet(p_general, index=False)
print(f"✔ BERT gerais → {p_general.resolve()}  ({len(df_general_bert)} linhas)")

✔ BERT complexidade → /home/dods/matrix/IC/Notebooks/meta/RTE_train_bert_features_complex.parquet  (12 linhas)
✔ BERT gerais → /home/dods/matrix/IC/Notebooks/meta/RTE_train_bert_features_general.parquet  (4 linhas)


### Teste de predição do Modelo

In [20]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "textattack/bert-base-uncased-RTE"  # checkpoint BERT já fine-tuned em RTE

# 1) Dataset: GLUE/RTE
ds = load_dataset("glue", "rte")

# 2) Tokenizer + tokenização de pares (BERT usa token_type_ids)
tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)

def preprocess(batch, max_len=128):
    return tok(
        batch["sentence1"], batch["sentence2"],
        truncation=True, padding="max_length", max_length=max_len
    )

enc = ds.map(preprocess, batched=True)

# 3) Formato para PyTorch
cols = ["input_ids", "attention_mask", "label"]
if "token_type_ids" in enc["validation"].features:
    cols.append("token_type_ids")
enc["validation"].set_format(type="torch", columns=cols)

val_loader = DataLoader(enc["validation"], batch_size=32, shuffle=False)

# 4) Modelo
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()

# 5) Avaliação
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask", "token_type_ids"]}
        labels = batch["label"].numpy()
        logits = model(**inputs).logits
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels)

import numpy as np
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

# 6) Métricas
acc = accuracy_score(y_true, y_pred)
cm  = confusion_matrix(y_true, y_pred, labels=[0,1])

id2label = model.config.id2label
target_names = [id2label.get(i, str(i)) for i in [0,1]]

print(f"Accuracy (validation): {acc:.4f}")
print("Confusion matrix [rows=true, cols=pred]:\n", cm)
print(classification_report(y_true, y_pred, target_names=target_names))


Map: 100%|██████████| 3000/3000 [00:00<00:00, 13995.44 examples/s]
/home/dods/miniconda3/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Accuracy (validation): 0.7256
Confusion matrix [rows=true, cols=pred]:
 [[126  20]
 [ 56  75]]
              precision    recall  f1-score   support

     LABEL_0       0.69      0.86      0.77       146
     LABEL_1       0.79      0.57      0.66       131

    accuracy                           0.73       277
   macro avg       0.74      0.72      0.72       277
weighted avg       0.74      0.73      0.72       277

